# LLM-Powered Term Sheet Reconciliation Walkthrough

Welcome to the interactive walkthrough notebook for the **LLM-Powered Term Sheet Reconciliation Engine**. 

This notebook contains all the core logic, prompts, and normalization layers of the project, organized in a logical execution sequence. You can run this notebook to reconcile mock trade records against unstructured PDF term sheets using Groq or OpenRouter (which provides free access to models like Google Gemini 2.5 Flash).

---

## Architecture Overview

The engine uses a robust **Two-Pass LLM Pipeline**:
1. **Pass 1 (Structured Extraction):** Parse unstructured PDF text and use the LLM to extract key bond parameters alongside verbatim "evidence quotes" for lineage validation. Structure is strictly validated using Pydantic.
2. **Programmatic Normalization:** Unify format variations in date formats, numbers (including Indian Lakhs/Crores multipliers), financial synonyms, and delimiter-separated list fields.
3. **Pass 2 (False Alert Adjudication):** For any remaining discrepancy, the LLM acts as an expert auditor to decide if it is a **False Alert** (formatting discrepancy) or a **True Mismatch** (material trade discrepancy). An **in-memory adjudication cache** is used to guarantee deterministic output and save API tokens.

## Step 1: Environment & Setup

We start by importing our dependencies, loading environment variables from a `.env` file, and setting up logging.

In [ ]:
import os
import re
import csv
import json
import time
import logging
from datetime import datetime
from typing import Optional, Tuple, List, Dict, Any, Union
from enum import Enum
import pandas as pd
import pdfplumber
from tabulate import tabulate
from pydantic import BaseModel, Field, field_validator, ValidationError
from dotenv import load_dotenv

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("NotebookReconciler")

# Load environment variables
load_dotenv()

print("✅ Environment dependencies imported successfully!")

## Step 2: Configuration constants

We define the active model, keys, target fields, and mappings matching `config.py`.

In [ ]:
# Config parameters
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Provider selection logic
if os.getenv("LLM_PROVIDER"):
    LLM_PROVIDER = os.getenv("LLM_PROVIDER").lower()
elif GROQ_API_KEY:
    LLM_PROVIDER = "groq"
else:
    raise ValueError("No LLM provider configured. Set GROQ_API_KEY in .env")

# Default model — Groq free-tier with 131k context (no truncation needed)
LLM_MODEL = os.getenv("LLM_MODEL", "meta-llama/llama-4-scout-17b-16e-instruct")

print(f"Active LLM Provider: {LLM_PROVIDER} | Active Model: {LLM_MODEL}")

# Target fields
RECONCILE_FIELDS = [
    "ISIN", "Issuer", "Maturity", "Notional", "Coupon", "Currency", 
    "SettlementDate", "DayCountFraction", "InterestPaymentDate", 
    "IssueDate", "IssueAmount", "IssuePrice", "NominalAmountPerBond", 
    "InterestPaymentFrequency", "BusinessDayConvention", 
    "BusinessDayLocation", "AmortizationType", "MinimumSubscription", "Parent"
]

TERM_SHEET_MAPPING = {
    "3.Term Sheet - IDBI.pdf": {
        "isin": "INE008A08U84",
        "booking_files": ["IDBI_Omni.csv", "IDBI_Omni.json"],
        "issuer_keyword": "IDBI"
    },
    "4.Term Sheet - Genel Energy.pdf": {
        "isin": "NO0010894330",
        "booking_files": ["Genel_Energy.csv", "Genel_Energy.json"],
        "issuer_keyword": "Genel"
    }
}

## Step 3: PDF Extractor

This module extracts clean text from multi-page PDFs using `pdfplumber`, normalizing excess whitespaces.

In [ ]:
def extract_text_from_pdf(file_path: str) -> str:
    """
    Extracts all text from a PDF file page-by-page.
    """
    try:
        text_pages = []
        with pdfplumber.open(file_path) as pdf:
            logger.info(f"Opened PDF: {file_path} ({len(pdf.pages)} pages)")
            for page_num, page in enumerate(pdf.pages, 1):
                page_text = page.extract_text()
                if page_text:
                    text_pages.append(page_text)
        
        full_text = "\n\n--- PAGE BREAK ---\n\n".join(text_pages)
        # Clean excessive spacing while preserving layout
        full_text = re.sub(r'[ \t]+', ' ', full_text)
        full_text = re.sub(r'\n{3,}', '\n\n', full_text)
        return full_text.strip()
    except Exception as e:
        logger.error(f"Failed to parse PDF {file_path}: {e}")
        raise

## Step 4: LLM Client

We wrap the Groq API in a client that supports structured JSON format output and exponential backoff retry policies.

In [ ]:
class LLMClient:
    """
    API client for Groq with retries and JSON enforcement.
    """
    def __init__(self):
        self.provider = LLM_PROVIDER
        self.model = LLM_MODEL
        self.max_retries = 3
        self.retry_delay = 2
        
        if self.provider == "groq":
            from groq import Groq
            self.client = Groq(api_key=GROQ_API_KEY)
        else:
            raise ValueError(f"Unknown provider: {self.provider}. Must be 'groq'.")
    
    def call_llm(self, system_prompt: str, user_prompt: str, json_mode: bool = True, temperature: float = 0.1) -> str:
        for attempt in range(1, self.max_retries + 1):
            try:
                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=temperature,
                    response_format={"type": "json_object"} if json_mode else None,
                    max_tokens=4096,
                )
                return response.choices[0].message.content
            except Exception as e:
                if attempt == self.max_retries:
                    raise
                time.sleep(self.retry_delay * attempt)

## Step 5: Pydantic Validation Models

We use Pydantic to enforce data structure. Every field extracted must contain a value and the exact verbatim snippet (`evidence`) matching the document.

In [ ]:
class ExtractedField(BaseModel):
    value: Optional[Union[str, float, int, List[str]]] = None
    evidence: str = "NOT_FOUND"

    @field_validator("evidence")
    @classmethod
    def validate_evidence(cls, v):
        if not v or len(v.strip()) < 2:
            raise ValueError("Evidence quote cannot be empty")
        return v.strip()

class TermSheetExtraction(BaseModel):
    ISIN: ExtractedField
    Issuer: ExtractedField
    Maturity: ExtractedField
    Notional: Optional[ExtractedField] = None
    Coupon: ExtractedField
    Currency: ExtractedField
    SettlementDate: Optional[ExtractedField] = None
    DayCountFraction: ExtractedField
    InterestPaymentDate: Optional[ExtractedField] = None
    IssueDate: ExtractedField
    IssueAmount: ExtractedField
    IssuePrice: Optional[ExtractedField] = None
    NominalAmountPerBond: ExtractedField
    InterestPaymentFrequency: ExtractedField
    BusinessDayConvention: ExtractedField
    BusinessDayLocation: Optional[ExtractedField] = None
    AmortizationType: ExtractedField
    MinimumSubscription: ExtractedField
    Parent: Optional[ExtractedField] = None

## Step 6: Pass 1 LLM Extraction

The extraction prompt contains specialized domain knowledge for financial formatting: 
- Translating "at par" $ightarrow$ `100`
- Converting Indian numbering like `Lakhs` and `Crores` to standard numeric string format.
- Directing Notional to remain null since it is trade-specific.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are an expert financial document analyst specializing in bond term sheets.
Your task is to extract specific fields from a term sheet document and return them in a strict JSON format.

CRITICAL REQUIREMENTS:
1. For EVERY field you extract, you MUST include an "evidence" key containing the EXACT verbatim 
   quote/snippet from the source text that was used to derive the value. Copy the text character-for-character.
2. If a field is not present in the document, set "value" to null and "evidence" to "NOT_FOUND".
3. Return ONLY valid JSON — no markdown, no explanation, no code blocks.

FIELD-SPECIFIC EXTRACTION RULES:
- ISIN: Look for an ISIN code (format: 2 letter country code + 9 alphanumeric characters + 1 check digit).
- Issuer: The legal entity name issuing the bond.
- Maturity: The maturity date. If the bond is perpetual, set value to "Perpetual".
- Coupon: The coupon/interest rate as a percentage, e.g., "10.75%".
- Currency: The 3-letter currency code (INR, USD, etc.).
- IssueDate: Look for Deemed Date of Allotment, Issue Date, etc. 
- IssueAmount: The total issue size converted to standard numeric string (Rs. 1,500 Crores = 15000000000).
- IssuePrice: "At par" or face value = 100.
- NominalAmountPerBond: Face value per bond (Rs. 10,00,000 = 1000000).
- BusinessDayLocation: Payment/holiday cities. Do NOT extract governing law cities (e.g. courts of Mumbai) as BusinessDayLocation.
- MinimumSubscription: Calculate value in currency units (5 Bonds * Rs. 10 Lakhs = 5000000).
- Notional: Must always be set to null (with evidence "NOT_FOUND").

Use this JSON schema:
{
    "ISIN": {"value": "<value>", "evidence": "<quote>"},
    "Issuer": {"value": "<value>", "evidence": "<quote>"},
    "Maturity": {"value": "<value>", "evidence": "<quote>"},
    "Notional": {"value": null, "evidence": "NOT_FOUND"},
    "Coupon": {"value": "<value>", "evidence": "<quote>"},
    "Currency": {"value": "<value>", "evidence": "<quote>"},
    "SettlementDate": {"value": "<value>", "evidence": "<quote>"},
    "DayCountFraction": {"value": "<value>", "evidence": "<quote>"},
    "InterestPaymentDate": {"value": "<value>", "evidence": "<quote>"},
    "IssueDate": {"value": "<value>", "evidence": "<quote>"},
    "IssueAmount": {"value": "<value>", "evidence": "<quote>"},
    "IssuePrice": {"value": "<value>", "evidence": "<quote>"},
    "NominalAmountPerBond": {"value": "<value>", "evidence": "<quote>"},
    "InterestPaymentFrequency": {"value": "<value>", "evidence": "<quote>"},
    "BusinessDayConvention": {"value": "<value>", "evidence": "<quote>"},
    "BusinessDayLocation": {"value": "<value>", "evidence": "<quote>"},
    "AmortizationType": {"value": "<value>", "evidence": "<quote>"},
    "MinimumSubscription": {"value": "<value>", "evidence": "<quote>"},
    "Parent": {"value": "<value>", "evidence": "<quote>"}
}"""

def extract_fields_from_termsheet(pdf_text: str, llm_client: LLMClient) -> dict:
    user_prompt = f"Extract all fields from the text:\n\n{pdf_text[:15000]}"
    try:
        raw = llm_client.call_llm(EXTRACTION_SYSTEM_PROMPT, user_prompt, json_mode=True)
        parsed = json.loads(raw)
        validated = TermSheetExtraction(**parsed)
        return {"status": "success", "data": validated.model_dump(), "errors": []}
    except Exception as e:
        logger.error(f"Extraction failed: {e}")
        return {"status": "failed", "data": {}, "errors": [str(e)]}

## Step 7: Booking System Ingestion

Reads structured booking transaction data from CSV or JSON files, flattening list parameters.

In [ ]:
def ingest_booking_file(file_path: str) -> List[Dict[str, Any]]:
    _, ext = os.path.splitext(file_path)
    ext = ext.lower()
    
    if ext == ".csv":
        df = pd.read_csv(file_path)
        if "BusinessDayLocation" in df.columns:
            df["BusinessDayLocation"] = df["BusinessDayLocation"].apply(
                lambda x: x.split("|") if isinstance(x, str) and "|" in x else [x] if pd.notna(x) else x
            )
        trades = df.to_dict(orient="records")
    else:
        with open(file_path, "r") as f:
            data = json.load(f)
        trades = data.get("trades", data) if isinstance(data, dict) else data
        
    for trade in trades:
        for key, val in list(trade.items()):
            if isinstance(val, list):
                trade[key] = "|".join(str(v) for v in val)
            elif pd.isna(val) or val is None:
                trade[key] = None
            else:
                trade[key] = str(val)
    return trades

## Step 8: Comparison Normalizers & Adjudication Caching

The core logic where initial comparison occurs. Standardizes date, number (with Indian multipliers), string synonyms, and business day lists.

Mismatches are sent to the LLM (at `temperature=0.0`) for classification, using the in-memory cache.

In [ ]:
FINANCIAL_SYNONYMS = {
    'semi annual': 'semiannual', 'semi-annual': 'semiannual', 'semi annually': 'semiannual',
    'bi annual': 'semiannual', 'bi-annual': 'semiannual', 'bullet repayment': 'bullet',
    'bullet maturity': 'bullet', 'at par': '100', 'at maturity': 'bullet'
}

def normalize_date(value: str) -> Optional[str]:
    if not value or value == "NOT_FOUND": return None
    for fmt in ["%Y-%m-%d", "%m/%d/%Y", "%d %B %Y", "%B %d, %Y", "%d-%m-%Y"]:
        try:
            return datetime.strptime(value.strip(), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    cleaned = re.sub(r'^(Expected to be|expected to be)\s+', '', value).strip()
    for fmt in ["%Y-%m-%d", "%m/%d/%Y", "%d %B %Y", "%B %d, %Y"]:
        try:
            return datetime.strptime(cleaned, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

def normalize_number(value: str) -> Optional[float]:
    if not value or value == "NOT_FOUND": return None
    val_lower = value.lower()
    cleaned = value.replace(',', '').replace(' ', '').replace('%', '')
    cleaned = re.sub(r'(?i)\b(Rs\.?|USD|EUR|GBP|INR|p\.a\.?)\b', '', cleaned)
    cleaned = re.sub(r'[\$\€\£\₹]', '', cleaned)
    cleaned = re.sub(r'(?i)\b(crores?|lakhs?)\b', '', cleaned)
    
    match = re.search(r'-?\d+\.?\d*', cleaned)
    if match:
        try:
            res = float(match.group())
            if 'crore' in val_lower: res *= 10000000
            elif 'lakh' in val_lower: res *= 100000
            return res
        except ValueError:
            return None
    return None

def normalize_string(value: str) -> Optional[str]:
    if not value or value == "NOT_FOUND": return None
    norm = value.lower().replace('-', ' ').strip()
    norm = re.sub(r'[\.,;:\'"()]+', '', norm)
    norm = re.sub(r'\s+', ' ', norm)
    return FINANCIAL_SYNONYMS.get(norm, norm)

def compare_fields(term_sheet_data: dict, booking_trades: list, booking_file: str, fields: list) -> list:
    results = []
    for trade in booking_trades:
        trade_id = trade.get("TradeID")
        isin = trade.get("ISIN", "UNKNOWN")
        for field_name in fields:
            ts_field = term_sheet_data.get(field_name, {})
            ts_value = ts_field.get("value") if isinstance(ts_field, dict) else None
            ts_evidence = ts_field.get("evidence", "") if isinstance(ts_field, dict) else ""
            booking_val = trade.get(field_name)
            
            # Handlers for missing fields
            if ts_value is None and ts_evidence == "NOT_FOUND":
                if booking_val is None or str(booking_val).lower() in ["", "none", "null"]:
                    results.append({"trade_id": trade_id, "isin": isin, "field_name": field_name, "status": "Match", "reason": "Both sources lack this field", "booking_file": booking_file})
                else:
                    results.append({"trade_id": trade_id, "isin": isin, "field_name": field_name, "status": "True Mismatch", "reason": f"Field is not present in the term sheet but contains value '{booking_val}' in the booking system", "booking_file": booking_file, "term_sheet_value": None, "term_sheet_evidence": "NOT_FOUND", "booking_value": booking_val})
                continue
            
            # Normalization comparisons
            ts_norm, ts_orig = (normalize_date(str(ts_value)), str(ts_value)) if field_name in ["Maturity", "SettlementDate", "InterestPaymentDate", "IssueDate"] else (str(normalize_number(str(ts_value))), str(ts_value)) if field_name in ["Coupon", "IssueAmount", "IssuePrice", "NominalAmountPerBond", "MinimumSubscription", "Notional"] else (normalize_string(str(ts_value)), str(ts_value))
            bk_norm, bk_orig = (normalize_date(str(booking_val)), str(booking_val)) if field_name in ["Maturity", "SettlementDate", "InterestPaymentDate", "IssueDate"] else (str(normalize_number(str(booking_val))), str(booking_val)) if field_name in ["Coupon", "IssueAmount", "IssuePrice", "NominalAmountPerBond", "MinimumSubscription", "Notional"] else (normalize_string(str(booking_val)), str(booking_val))
            
            # Programmatic sorting delimiter lists
            if field_name == "BusinessDayLocation" and ts_norm and bk_norm:
                ts_parts = sorted(set(p.strip() for p in re.split(r'[|,;]+', ts_norm) if p.strip()))
                bk_parts = sorted(set(p.strip() for p in re.split(r'[|,;]+', bk_norm) if p.strip()))
                if ts_parts == bk_parts: ts_norm, bk_norm = "|".join(ts_parts), "|".join(bk_parts)
            
            # Minimum Subscription bonds multiplier conversion check
            if field_name == "MinimumSubscription" and ts_value:
                bond_match = re.search(r'(\d+)\s*(?:bonds?|debt\s+securities)', str(ts_value).lower())
                if bond_match:
                    bond_count = int(bond_match.group(1))
                    nominal_val = term_sheet_data.get("NominalAmountPerBond", {}).get("value")
                    nominal_num = normalize_number(str(nominal_val)) if nominal_val else None
                    if nominal_num:
                        calculated_min_sub = bond_count * nominal_num
                        ts_norm = str(calculated_min_sub)

            # Notional comparison
            if field_name == "Notional" and ts_norm and bk_norm:
                issue_amt_val = term_sheet_data.get("IssueAmount", {}).get("value")
                if issue_amt_val:
                    issue_norm = str(normalize_number(str(issue_amt_val)))
                    if ts_norm == issue_norm and float(bk_norm) <= float(ts_norm):
                        results.append({"trade_id": trade_id, "isin": isin, "field_name": field_name, "status": "Match", "reason": f"Trade notional ({bk_orig}) is within the total issue size ({ts_orig})", "booking_file": booking_file, "term_sheet_value": ts_orig, "term_sheet_evidence": ts_evidence, "booking_value": bk_orig})
                        continue

            if ts_norm == bk_norm:
                results.append({"trade_id": trade_id, "isin": isin, "field_name": field_name, "status": "Match", "reason": "Values match after normalization", "booking_file": booking_file, "term_sheet_value": ts_orig, "term_sheet_evidence": ts_evidence, "booking_value": bk_orig})
            else:
                results.append({"trade_id": trade_id, "isin": isin, "field_name": field_name, "status": "Mismatch", "reason": "Normalized values differ", "booking_file": booking_file, "term_sheet_value": ts_orig, "term_sheet_evidence": ts_evidence, "booking_value": bk_orig})
    return results

## Step 9: Pass 2 Adjudicator (with Caching)

In [ ]:
FALSE_ALERT_SYSTEM_PROMPT = """You are a financial reconciliation expert.
Your task is to determine whether a discrepancy between a term sheet and a booking system is a TRUE MISMATCH or a FALSE ALERT.

A FALSE ALERT means the values are semantically equivalent but differ in layout/wording formatting.
A TRUE MISMATCH means the values are genuinely different.

Return JSON matching:
{
    "status": "False Alert" | "True Mismatch",
    "reason": "<explanation>"
}"""

def analyze_false_alerts(mismatches: list, llm_client: LLMClient) -> list:
    resolved_results = []
    adjudication_cache = {}
    
    for mismatch in mismatches:
        if mismatch["status"] != "Mismatch":
            resolved_results.append(mismatch)
            continue
        
        cache_key = (mismatch["field_name"], mismatch["term_sheet_value"], mismatch["booking_value"])
        if cache_key in adjudication_cache:
            cached = adjudication_cache[cache_key]
            mismatch["status"] = cached["status"]
            mismatch["reason"] = cached["reason"]
            resolved_results.append(mismatch)
            continue
            
        user_prompt = f"FIELD: {mismatch['field_name']}\nTS: {mismatch['term_sheet_value']}\nTS Quote: {mismatch['term_sheet_evidence']}\nBooking: {mismatch['booking_value']}"
        try:
            raw = llm_client.call_llm(FALSE_ALERT_SYSTEM_PROMPT, user_prompt, json_mode=True, temperature=0.0)
            res = json.loads(raw)
            mismatch["status"] = res.get("status", "True Mismatch")
            mismatch["reason"] = res.get("reason", "Adjudicated by LLM")
            
            adjudication_cache[cache_key] = {"status": mismatch["status"], "reason": mismatch["reason"]}
        except Exception as e:
            mismatch["status"] = "True Mismatch"
            mismatch["reason"] = f"LLM failed: {e}"
        resolved_results.append(mismatch)
    return resolved_results

## Step 10: Interactive E2E Reconciliation Loop

You can now run the cells below to execute the full workflow on your environment data files under `Assignment/`!

In [ ]:
# Orchestrator Demo
try:
    client = LLMClient()
    all_results = []
    
    for pdf_filename, mapping in TERM_SHEET_MAPPING.items():
        pdf_path = os.path.join('Assignment', pdf_filename)
        if not os.path.exists(pdf_path):
            print(f'📄 Skipped {pdf_filename} - file not present in local path')
            continue
            
        print(f'\nProcessing Term Sheet: {pdf_filename}...')
        pdf_text = extract_text_from_pdf(pdf_path)
        extraction = extract_fields_from_termsheet(pdf_text, client)
        
        if extraction['status'] != 'success':
            print(f'❌ Extraction failed for {pdf_filename}')
            continue
            
        ts_data = extraction['data']
        
        for booking_filename in mapping['booking_files']:
            booking_path = os.path.join('Assignment', booking_filename)
            if not os.path.exists(booking_path):
                continue
                
            print(f'  -> Comparing against {booking_filename}...')
            booking_trades = ingest_booking_file(booking_path)
            raw_compares = compare_fields(ts_data, booking_trades, booking_filename, RECONCILE_FIELDS)
            final_resolves = analyze_false_alerts(raw_compares, client)
            all_results.extend(final_resolves)
            
    # Print Summary Table if results exist
    if all_results:
        status_counts = {}
        for r in all_results:
            status = r.get('status', 'Unknown')
            status_counts[status] = status_counts.get(status, 0) + 1
            
        print('\n' + '=' * 60)
        print(' INTERACTIVE RECONCILIATION ROLLUP SUMMARY')
        print('=' * 60)
        for status, count in sorted(status_counts.items()):
            print(f'  {status:20s}: {count}')
        print(f'  {"TOTAL":20s}: {len(all_results)}')
        print('=' * 60)
        
        # Show some discrepancies
        discrepancies = [r for r in all_results if r.get('status') in ['True Mismatch', 'False Alert']]
        if discrepancies:
            headers = ['Trade', 'ISIN', 'Field', 'TermSheet', 'Booking', 'Status', 'Reason']
            table_data = [[
                d.get('trade_id'), d.get('isin'), d.get('field_name'), 
                str(d.get('term_sheet_value', ''))[:25], str(d.get('booking_value', ''))[:25], 
                d.get('status'), str(d.get('reason', ''))[:45]
            ] for d in discrepancies[:15]]
            print('\nSample Discrepancies Adjudicated:')
            print(tabulate(table_data, headers=headers, tablefmt='grid'))
except Exception as err:
    print(f'⚠️ Error running interactive demo cell: {err}')